# Muster Lab — Try to defend

**Choose a Watchline. Replay the incident. See what becomes reachable.**

A Watchline has three defensive roles:

- **Vedette — detect / understand.** Creates defensive knowledge.
- **Picket — prevent / intercept.** Stops an attempted transition.
- **Reserve — respond / recover.** Changes state after detection and escalation.

This notebook is a presentation and experiment-selection layer over the Go engine.
Python selects controls and explains returned traces; it does **not** decide whether
events apply or controls fire.

Start with the **Toy incident** and try to keep the attacker off the node. When you
want the counterintuitive case, load **Want something counterintuitive?**


In [9]:
from pathlib import Path
import json
import re
import subprocess
from html import escape
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError(
        "This notebook needs ipywidgets. Install it with: python -m pip install ipywidgets"
    ) from exc


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "go.mod").exists() and (candidate / "engine").exists():
            return candidate
    raise RuntimeError("Could not find Muster repository root (go.mod + engine/).")


ROOT = find_repo_root()


def snake(name):
    s1 = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    return re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s1).lower()


def normalize(value):
    if isinstance(value, dict):
        return {snake(k): normalize(v) for k, v in value.items()}
    if isinstance(value, list):
        return [normalize(v) for v in value]
    return value


def run_muster(scenario, controls, enabled):
    cmd = [
        "go", "run", ".", "replay",
        "--scenario", str(scenario),
        "--controls", str(controls),
        "--json",
    ]
    if enabled is not None:
        cmd.extend(["--only-controls", ",".join(enabled)])

    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or proc.stdout.strip())

    doc = normalize(json.loads(proc.stdout))
    result = doc["result"]
    result["control_set_id"] = doc.get("control_set_id")
    result["enabled_controls"] = list(enabled or [])
    return result


In [10]:
# Human-facing UX metadata only. Engine semantics remain in YAML + Go.
SCENARIOS = {
    "Toy incident": {
        "scenario": ROOT / "examples/notebook/toy-jackpot.yaml",
        "controls": ROOT / "examples/notebook/toy-controls.yaml",
        "adverse": ["attacker:node-access"],
        "budget": 4,
        "fixed": [],
        "event_labels": {
            "worker-code-execution": "Gain code execution on worker",
            "cluster-credential-theft": "Steal cluster credential",
            "kubernetes-api-discovery-from-worker": "Discover Kubernetes API",
            "stolen-credential-replay": "Replay stolen credential",
            "privileged-workload-creation": "Create privileged workload",
            "node-access": "Reach the node",
        },
        "fact_labels": {
            "attacker:node-access": "attacker reached the node",
            "attacker:worker-access": "worker access",
            "attacker:cluster-credential": "stolen cluster credential",
            "attacker:cluster-knowledge": "cluster knowledge",
            "attacker:cluster-session": "cluster session",
            "workload:privileged-hostpath": "privileged workload",
            "alert:credential-theft": "credential-theft alert",
            "alert:k8s-discovery": "Kubernetes-discovery alert",
            "reviewed:cluster-integrity": "cluster integrity reviewed",
            "critical:cluster-integrity": "cluster compromise marked critical",
            "escalated:cluster-integrity": "cluster response escalated",
            "escalated:worker-containment": "worker containment escalated",
        },
        "choices": [
            {
                "key":"V1","role":"Vedette","label":"Detect credential theft",
                "ids":["worker-credential-theft"],
                "help":"Notice the credential theft and preserve an alert for later defenders."
            },
            {
                "key":"V2","role":"Vedette","label":"Detect Kubernetes discovery",
                "ids":["worker-k8s-discovery"],
                "help":"Notice later cluster discovery; this can feed the credential-revocation response."
            },
            {
                "key":"P1","role":"Picket","label":"Stop credential replay",
                "ids":["block-credential-replay"],
                "help":"Prevent the stolen credential from creating a cluster session."
            },
            {
                "key":"P2","role":"Picket","label":"Block privileged workload",
                "ids":["block-privileged-workload"],
                "help":"Reject the privileged workload before it can lead to node access."
            },
            {
                "key":"R1","role":"Reserve","label":"Revoke stolen credential",
                "ids":["review-k8s-discovery","assess-cluster-integrity","escalate-cluster-integrity","revoke-cluster-credential"],
                "help":"Use the later discovery signal to review, escalate, and revoke the stolen credential."
            },
            {
                "key":"R2","role":"Reserve","label":"Isolate worker early",
                "ids":["escalate-worker-containment","isolate-worker"],
                "help":"Contain the worker immediately after credential theft. Useful — but perhaps not harmless."
            },
        ],
        "presets": {
            "Start clean": [],
            "Strong but brittle": ["V1","V2","R1"],
            "Diverse prevention": ["V1","P1","P2"],
            "Want something counterintuitive?": ["V1","V2","R1","R2"],
        },
    },
    "HF-inspired public-record abstraction": {
        "scenario": ROOT / "examples/hf-july-2026/scenario.yaml",
        "controls": ROOT / "examples/notebook/hf-controls.yaml",
        "adverse": ["attacker:node-root","attacker:internal-network-access","attacker:cluster-admin"],
        "budget": None,
        "fixed": [],
        "event_labels": {
            "dataset-local-file-read": "Read local files through dataset pipeline",
            "dataset-template-rce": "Gain code execution on worker",
            "k8s-api-discovery": "Discover Kubernetes API",
            "cloud-metadata-role-replay": "Replay cloud role",
            "privileged-hostpath-pod": "Create privileged host-mounted pod",
            "node-root": "Gain root on node",
            "cluster-secret-harvest": "Harvest cluster secrets",
            "mesh-enrollment": "Pivot onto internal network",
            "service-connector-discovery": "Discover internal service connector",
            "shared-connector-credential-replay": "Replay shared connector credential",
        },
        "fact_labels": {
            "attacker:node-root": "attacker gained node root",
            "attacker:internal-network-access": "attacker reached the internal network",
            "attacker:cluster-admin": "attacker gained cluster-admin",
            "attacker:worker-access": "worker access",
            "attacker:worker-secrets": "worker secrets",
            "attacker:cluster-knowledge": "cluster knowledge",
            "attacker:privileged-hostpath": "privileged host-mounted workload",
            "attacker:cluster-secrets": "cluster secrets",
            "attacker:connector-catalog": "service-connector catalog",
            "alert:worker-compromise": "worker-compromise alert",
            "reviewed:worker-compromise": "worker compromise reviewed",
            "critical:worker-compromise": "worker compromise marked critical",
            "escalated:worker-isolation": "worker isolation escalated",
        },
        "choices": [
            {
                "key":"V1","role":"Vedette","label":"Detect worker compromise",
                "ids":["correlate-worker-compromise"],
                "help":"Illustrative correlated signal; not a reconstruction of Hugging Face's exact alerting."
            },
            {
                "key":"P1","role":"Picket","label":"Block privileged host-mounted pod",
                "ids":["reject-privileged-hostpath"],
                "help":"Cuts the node-root and internal-network branch."
            },
            {
                "key":"P2","role":"Picket","label":"Scope connector credential",
                "ids":["reject-cross-cluster-connector-replay"],
                "help":"Cuts the separate cluster-admin connector branch."
            },
            {
                "key":"R1","role":"Reserve","label":"Page on-call and isolate worker",
                "ids":["review-worker-compromise","assess-worker-compromise-critical","escalate-worker-isolation","isolate-worker"],
                "help":"Review the alert, mark it critical, escalate, then isolate the compromised worker."
            },
        ],
        "presets": {
            "Observed signal / no page": ["V1"],
            "Admission policy": ["V1","P1"],
            "Connector scoping": ["V1","P2"],
            "Critical page + isolate": ["V1","R1"],
            "Layered": ["V1","P1","P2","R1"],
        },
    },

    "Anthropic PyPI incident": {
        "scenario": ROOT / "examples/anthropic-pypi-2026/scenario.yaml",
        "controls": ROOT / "examples/anthropic-pypi-2026/controls.yaml",
        "adverse": ["attacker:vendor-database-access"],
        "budget": None,
        "fixed": [],
        "scenario_note": (
            "<b>Public-record abstraction.</b> A cyber-evaluation run reached "
            "the real internet, published a malicious PyPI package, and ultimately "
            "accessed a third-party security vendor's live database."
        ),
        "event_labels": {
            "discover-unregistered-package": "Notice an unregistered package name",
            "reach-public-internet": "Reach the real public internet",
            "create-public-registry-account": "Create a public package-registry account",
            "publish-malicious-package": "Publish malicious package to PyPI",
            "third-party-scanner-installs-package": "Security vendor scanner installs package",
            "scanner-credential-exposure": "Scanner exposes vendor credential",
            "vendor-database-access": "Access vendor's live database",
            "registry-removes-package": "PyPI removes malicious package",
        },
        "fact_labels": {
            "attacker:vendor-database-access": "attacker accessed the vendor's live database",
            "agent:package-name-known": "unregistered package name known",
            "agent:public-internet-access": "public internet access",
            "agent:pypi-account": "public registry account",
            "public:malicious-package": "malicious package publicly available",
            "third-party:package-executed": "package executed in vendor scanner",
            "attacker:vendor-credential": "vendor credential exposed",
            "public:package-removed": "malicious package removed from registry",
            "alert:unexpected-internet": "unexpected-internet alert",
            "reviewed:unexpected-internet": "unexpected internet access reviewed",
            "critical:eval-boundary": "evaluation boundary failure marked critical",
            "escalated:eval-termination": "evaluation termination escalated",
            "alert:vendor-credential-exposure": "vendor credential-exposure alert",
            "reviewed:vendor-credential-exposure": "credential exposure reviewed",
            "critical:vendor-credential": "credential exposure marked critical",
            "escalated:vendor-credential-revocation": "credential revocation escalated",
        },
        "choices": [
            {"key":"V1","role":"Vedette","label":"Detect unexpected internet access",
             "ids":["detect-unexpected-internet"],
             "help":"Notice that a supposedly sealed cyber evaluation has reached the public internet."},
            {"key":"P1","role":"Picket","label":"Block evaluation internet access",
             "ids":["block-eval-internet"],
             "help":"Stop the evaluation at its first real-world boundary."},
            {"key":"P2","role":"Picket","label":"Block public package publication",
             "ids":["block-public-package-publish"],
             "help":"Allow internet access but prevent the evaluation identity from publishing a package."},
            {"key":"R1","role":"Reserve","label":"Terminate evaluation after boundary alert",
             "ids":["review-unexpected-internet","assess-eval-boundary-critical",
                    "escalate-eval-termination","terminate-evaluation"],
             "help":"Use the unexpected-internet alert to review, escalate, and stop the evaluation."},
            {"key":"P3","role":"Picket","label":"Isolate scanner from reusable credentials",
             "ids":["isolate-scanner-credentials"],
             "help":"The package can execute, but the third-party scanner cannot leak a credential usable against production."},
            {"key":"V2","role":"Vedette","label":"Detect scanner credential exposure",
             "ids":["detect-scanner-credential-exposure"],
             "help":"Notice credential exposure inside the third-party vendor."},
            {"key":"R2","role":"Reserve","label":"Revoke exposed vendor credential",
             "ids":["review-vendor-credential-exposure","assess-vendor-credential-critical",
                    "escalate-vendor-credential-revocation","revoke-vendor-credential"],
             "help":"After detection, revoke the vendor credential before it can be used against the live database."},
        ],
        "presets": {
            "Observed incident shape": [],
            "Monitor evaluation only": ["V1"],
            "Seal evaluation egress": ["P1"],
            "Block registry write": ["P2"],
            "Detect + terminate evaluation": ["V1","R1"],
            "Sandbox third-party scanner": ["P3"],
            "Vendor detect + revoke": ["V2","R2"],
            "Layered across boundaries": ["V1","P2","R1","P3","V2","R2"],
        },
    },
}


In [11]:
def entry_controls(entry):
    return [c for c in (entry.get("controls") or []) if c.get("matched")]


def acted_controls(entry):
    return [c for c in entry_controls(entry) if c.get("acted")]


def state_delta(entry):
    before = set(entry.get("before") or [])
    after = set(entry.get("after") or [])
    return sorted(after - before), sorted(before - after)


def terminal_adverse(run, facts):
    terminal = set(run.get("terminal_state") or [])
    return [fact for fact in facts if fact in terminal]


def display_event(event_id):
    return cfg().get("event_labels", {}).get(event_id, event_id.replace("-", " ").title())


def display_fact(fact):
    return cfg().get("fact_labels", {}).get(fact, fact)


def display_control(control):
    """Prefer notebook-facing labels, but preserve engine IDs in details."""
    control_id = control.get("control_id", "")
    for choice in cfg()["choices"]:
        if control_id in choice["ids"]:
            return choice["label"]
    return control_id.replace("-", " ").title()


def defensive_activity(run):
    acted = [
        control
        for entry in run["trace"]
        for control in acted_controls(entry)
    ]
    detected = any(
        str(c.get("role", "")).lower() == "vedette"
        or str(c.get("action", "")).lower() == "observe"
        for c in acted
    )
    prevented = any(
        str(c.get("role", "")).lower() == "picket"
        or str(c.get("action", "")).lower() == "block"
        for c in acted
    ) or any(
        str(entry.get("status", "")).lower() == "blocked"
        for entry in run["trace"]
    )
    responded = any(
        str(c.get("role", "")).lower() == "reserve"
        or str(c.get("action", "")).lower() == "respond"
        for c in acted
    )
    return {
        "detected": detected,
        "prevented": prevented,
        "responded": responded,
        "acted": acted,
    }


def classify_outcome(run, adverse):
    """
    Human-facing outcome class. This does not affect replay semantics.

    The headline describes the whole run; the activity badges beneath it preserve
    partial successes such as one Picket blocking a branch while another adverse
    branch remains reachable.
    """
    bad = terminal_adverse(run, adverse)
    activity = defensive_activity(run)

    if bad:
        if activity["responded"]:
            label = "ADVERSE DESPITE RESPONSE"
            explanation = "A Reserve acted, but at least one modeled adverse outcome remained reachable."
        elif activity["detected"]:
            label = "DETECTED, NOT CONTAINED"
            explanation = "A Vedette created defensive knowledge, but the deployed Watchline did not contain every adverse path."
        elif activity["prevented"]:
            label = "PARTIALLY PREVENTED, ADVERSE REMAINS"
            explanation = "A Picket stopped part of the incident, but another modeled adverse path survived."
        else:
            label = "UNDETECTED & UNCONTAINED"
            explanation = "No deployed defense acted before the modeled adverse outcome was reached."
    else:
        if activity["responded"]:
            label = "RESPONDED & CONTAINED"
            explanation = "The Watchline detected the incident and a Reserve changed state before adverse impact."
        elif activity["prevented"]:
            label = "PREVENTED"
            explanation = "A Picket stopped the incident before any modeled adverse outcome was reached."
        elif activity["detected"]:
            # Rare but honest: the scenario may end safely for reasons other than containment.
            label = "DETECTED; NO ADVERSE IMPACT"
            explanation = "A Vedette detected activity and the run ended without a modeled adverse outcome."
        else:
            label = "NO ADVERSE IMPACT"
            explanation = "The run ended without a modeled adverse outcome and without defensive action."

    return label, explanation, bad, activity


def first_divergence(previous, current):
    if not previous:
        return None

    a = {e["event_id"]: e for e in previous["trace"]}
    b = {e["event_id"]: e for e in current["trace"]}
    order = []
    for run in (previous, current):
        for entry in run["trace"]:
            if entry["event_id"] not in order:
                order.append(entry["event_id"])

    for event_id in order:
        ea, eb = a.get(event_id), b.get(event_id)
        if ea is None or eb is None:
            return event_id, ea, eb

        sig_a = (
            ea.get("status"),
            tuple(sorted(ea.get("after") or [])),
            tuple(sorted(c["control_id"] for c in acted_controls(ea))),
        )
        sig_b = (
            eb.get("status"),
            tuple(sorted(eb.get("after") or [])),
            tuple(sorted(c["control_id"] for c in acted_controls(eb))),
        )
        if sig_a != sig_b:
            return event_id, ea, eb

    return None


def render_run(run, adverse, previous=None):
    outcome, explanation, bad, activity = classify_outcome(run, adverse)

    activity_badges = []
    activity_badges.append(
        f"<span style='padding:3px 7px;border:1px solid #aaa;border-radius:999px'>"
        f"Vedette: {'acted' if activity['detected'] else '—'}</span>"
    )
    activity_badges.append(
        f"<span style='padding:3px 7px;border:1px solid #aaa;border-radius:999px'>"
        f"Picket: {'acted' if activity['prevented'] else '—'}</span>"
    )
    activity_badges.append(
        f"<span style='padding:3px 7px;border:1px solid #aaa;border-radius:999px'>"
        f"Reserve: {'acted' if activity['responded'] else '—'}</span>"
    )

    cards = []
    for entry in run["trace"]:
        added, removed = state_delta(entry)
        acted = acted_controls(entry)

        acted_text = ", ".join(
            f"{escape(display_control(c))} "
            f"<code style='color:#666'>[{escape(c['control_id'])}]</code>"
            for c in acted
        ) or "—"

        unsatisfied = []
        for cond in entry.get("unsatisfied") or []:
            fact = cond.get("fact") or cond.get("Fact") or "?"
            unsatisfied.append(display_fact(str(fact)))

        why = (
            f"<div><b>Missing prerequisite:</b> {escape(', '.join(unsatisfied))}</div>"
            if unsatisfied else ""
        )

        friendly_added = ", ".join(display_fact(f) for f in added) or "—"
        friendly_removed = ", ".join(display_fact(f) for f in removed) or "—"
        status = str(entry.get("status", "")).upper()

        cards.append(f"""
        <details style="border:1px solid #bbb;border-radius:8px;padding:8px 10px;margin:6px 0">
          <summary style="cursor:pointer">
            <b>{escape(display_event(entry['event_id']))}</b>
            <span style="float:right;text-transform:uppercase">{escape(status)}</span>
            <div style="font-size:0.78em;color:#666;margin-top:2px">
              <code>{escape(entry['event_id'])}</code>
            </div>
          </summary>
          <div style="margin-top:8px;font-size:0.92em">
            {why}
            <div><b>Defense acted:</b> {acted_text}</div>
            <div><b>State gained:</b> {escape(friendly_added)}</div>
            <div><b>State removed:</b> {escape(friendly_removed)}</div>
          </div>
        </details>
        """)

    diff_html = ""
    divergence = first_divergence(previous, run)
    if divergence:
        event_id, before, after = divergence
        a_status = before.get("status") if before else "missing"
        b_status = after.get("status") if after else "missing"
        diff_html = f"""
        <div style="margin-top:12px;padding:10px;border-left:4px solid #888">
          <b>First causal divergence:</b> {escape(display_event(event_id))}<br>
          <code>{escape(event_id)}</code><br>
          previous: {escape(str(a_status))} → current: {escape(str(b_status))}
        </div>
        """
    elif previous:
        diff_html = """
        <div style="margin-top:12px;padding:10px;border-left:4px solid #888">
          <b>No trajectory divergence.</b>
        </div>
        """

    adverse_text = ", ".join(display_fact(f) for f in bad) or "none"

    return HTML(f"""
    <div style="font-family:system-ui,sans-serif">
      <div style="padding:12px 14px;border:1px solid #999;border-radius:10px;margin-bottom:10px">
        <div style="font-size:1.2em;font-weight:750">{escape(outcome)}</div>
        <div style="margin-top:3px;color:#555">{escape(explanation)}</div>
        <div style="display:flex;gap:6px;flex-wrap:wrap;margin-top:9px">
          {''.join(activity_badges)}
        </div>
        <div style="margin-top:9px;font-size:0.88em;color:#666">
          <b>Modeled adverse impact:</b> {escape(adverse_text)}
        </div>
      </div>
      {''.join(cards)}
      {diff_html}
    </div>
    """)


In [12]:
scenario_dropdown = widgets.Dropdown(
    options=list(SCENARIOS), value="Toy incident", description="Scenario:",
    layout=widgets.Layout(width="560px")
)
preset_dropdown = widgets.Dropdown(description="Preset:", layout=widgets.Layout(width="560px"))
controls_box = widgets.VBox()
budget_label = widgets.HTML()
run_button = widgets.Button(description="Replay", button_style="primary", icon="play")
compare_previous = widgets.Checkbox(value=True, description="Compare with previous replay")
status = widgets.HTML()
scenario_note = widgets.HTML()
output = widgets.Output()

_state = {"previous": None}
_checkboxes = {}


ROLE_LEGEND = widgets.HTML("""
<div style="margin:6px 0 12px 0">
  <div style="display:flex;gap:12px;flex-wrap:wrap">
    <span><b>Vedette</b> — detect / understand</span>
    <span><b>Picket</b> — prevent / intercept</span>
    <span><b>Reserve</b> — respond / recover</span>
  </div>

  <div style="margin-top:7px;color:#666">
    <b>Broad Vedettes. Selective Pickets. Robust Reserves.</b>
  </div>
</div>
""")


def cfg():
    return SCENARIOS[scenario_dropdown.value]


def selected_keys():
    return [key for key, cb in _checkboxes.items() if cb.value]


def expanded_ids(keys):
    by_key = {c["key"]: c for c in cfg()["choices"]}
    ids = list(cfg()["fixed"])
    for key in keys:
        ids.extend(by_key[key]["ids"])
    return list(dict.fromkeys(ids))


def update_budget(*_):
    used = len(selected_keys())
    budget = cfg()["budget"]

    if budget is None:
        budget_label.value = """
        <div style="margin:8px 0 12px 0;color:#666">
          <b>No artificial defense budget.</b>
          Explore how different Watchlines cut the incident.
        </div>
        """
        return

    over = used > budget
    remaining = max(0, budget - used)

    if over:
        count = f"<b>{used}/{budget} selected — over budget</b>"
    else:
        count = (
            f"<b>{used}/{budget} selected</b>"
            f" &nbsp;·&nbsp; {remaining} remaining"
        )

    budget_label.value = f"""
    <div style="
        margin:8px 0 14px 0;
        padding:10px 12px;
        border-left:4px solid #888;
    ">
      <div>{count}</div>
      <div style="margin-top:4px">
        <b>You cannot defend every transition.</b>
        Where do you place your Watchline?
      </div>
    </div>
    """


def apply_preset(*_):
    keys = set(cfg()["presets"][preset_dropdown.value])
    for key, cb in _checkboxes.items():
        cb.value = key in keys
    update_budget()


def rebuild_controls(*_):
    global _checkboxes
    _checkboxes = {}
    groups = []
    note = cfg().get("scenario_note", "")
    scenario_note.value = (
        f"<div style='padding:8px 10px;border-left:4px solid #aaa;margin:4px 0 10px 0'>{note}</div>"
        if note else ""
    )

    if scenario_dropdown.value == "Toy incident":
    scenario_note.value = """
    <div style="margin:4px 0 10px 0;color:#666">
      Pickets are chosen with knowledge of this incident, so a correctly placed
      Picket will look unusually strong. The harder question is what survives
      when the next attack takes a path you did not anticipate.
    </div>
    """

    role_subtitles = {
        "Vedette": "Creates defensive knowledge. A Vedette alone does not stop the attacker.",
        "Picket": "Stops an attempted action before it changes incident state.",
        "Reserve": "Acts after detection / escalation to change the defensive situation.",
    }

    for role in ("Vedette", "Picket", "Reserve"):
        children = [
            widgets.HTML(
                f"<b>{role}</b>"
                f"<div style='color:#666;font-size:0.9em'>{role_subtitles[role]}</div>"
            )
        ]
        for choice in [c for c in cfg()["choices"] if c["role"] == role]:
            cb = widgets.Checkbox(
                value=False,
                description=f"{choice['key']} · {choice['label']}",
                indent=False,
                layout=widgets.Layout(width="680px"),
            )
            cb.observe(update_budget, names="value")
            _checkboxes[choice["key"]] = cb
            children.extend([
                cb,
                widgets.HTML(
                    f"<span style='color:#666;margin-left:24px'>"
                    f"{escape(choice['help'])}</span>"
                ),
            ])
        groups.append(widgets.VBox(children))

    controls_box.children = groups
    preset_dropdown.options = list(cfg()["presets"])
    preset_dropdown.value = list(cfg()["presets"])[0]
    apply_preset()
    _state["previous"] = None
    status.value = ""
    with output:
        clear_output()


def replay_clicked(_):
    keys = selected_keys()
    budget = cfg()["budget"]

    if budget is not None and len(keys) > budget:
        status.value = "<b>Over budget.</b> Remove a defense or load a preset."
        return

    enabled = expanded_ids(keys)
    status.value = "Running Muster…"

    try:
        run = run_muster(cfg()["scenario"], cfg()["controls"], enabled)
    except Exception as exc:
        status.value = f"<b>Replay failed:</b> {escape(str(exc))}"
        return

    previous = _state["previous"] if compare_previous.value else None

    with output:
        clear_output()
        display(render_run(run, cfg()["adverse"], previous=previous))

    _state["previous"] = run
    status.value = (
        f"<b>Engine run complete.</b> "
        f"<span style='color:#666'>Expanded engine controls: "
        f"{escape(', '.join(enabled) or 'none')}</span>"
    )


scenario_dropdown.observe(rebuild_controls, names="value")
preset_dropdown.observe(apply_preset, names="value")
run_button.on_click(replay_clicked)
rebuild_controls()

display(widgets.VBox([
    widgets.HTML("<h3>Compose a Watchline</h3>"),
    ROLE_LEGEND,
    scenario_dropdown,
    scenario_note,
    preset_dropdown,
    budget_label,
    controls_box,
    widgets.HBox([run_button, compare_previous]),
    status,
    output,
]))


## Suggested co-author walkthrough

1. **Toy → Start clean**: **UNDETECTED & UNCONTAINED**.
2. Turn on only a Vedette: **DETECTED, NOT CONTAINED**.
3. Try to defend under the toy budget.
4. **Strong but brittle**: **RESPONDED & CONTAINED**.
5. **Want something counterintuitive?**: **ADVERSE DESPITE RESPONSE**.
6. Expand the first divergence and inspect the lost defensive opportunity.
7. Switch to **HF-inspired public-record abstraction** and compare branch-specific Pickets with upstream response.
8. Finish with **Anthropic PyPI incident**:
   - **Monitor evaluation only**
   - **Seal evaluation egress**
   - **Sandbox third-party scanner**
   - **Vendor detect + revoke**

The Anthropic case adds a different lesson: one incident can cross several
defensive owners. The Watchline spans the evaluation operator, a public package
registry, and an unrelated third-party security vendor.

Both real scenarios are deliberately compressed public-record abstractions, not forensic reconstructions.


## Presentation-layer rule

If Python ever needs to decide whether an event *should* apply, whether a control
*should* fire, whether suppression is satisfied, or how effects mutate state,
stop. That logic belongs in Go.

The notebook is allowed to select controls, invoke Muster, compare returned traces,
compute display-only before/after diffs, and render the result.
